# StreamVault back-catalog analysis — the narrative

**This notebook contains no analysis.** Every number below is computed by a function
in `pipeline/` and every table is an artefact written by `run_pipeline.py`.
That is deliberate: if a figure existed only in a cell here, it would not survive
Adjoa's rerun next quarter, and nobody could tell which cells had been run in which
order to produce it.

So read this as a guided tour of work that happens elsewhere. To change a number,
change the code or the settings — not a cell.

**Start the pipeline first**, if you have not already:

```
python run_pipeline.py
```

The one-page answer is in [`reports/memo.md`](../reports/memo.md). This notebook is
for someone who wants to see how it was reached.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# This notebook lives in explore/, so the repository root is one level up.
# It imports the pipeline and computes nothing of its own: every number below
# comes from the same code run_pipeline.py runs, so the narrative can never
# disagree with the artefacts (ADR-007).
sys.path.insert(0, str(Path.cwd().parent))

from pipeline.config import load_settings
from pipeline.extract import load_fetch_log, load_ledger, reconciliation_summary

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

settings = load_settings()
PROCESSED = settings.path(settings["paths"]["processed_dir"])
INTERIM = settings.path(settings["paths"]["interim_dir"])
REPORTS = settings.path(settings["paths"]["reports_dir"])
print("settings loaded and validated")

settings loaded and validated


## As of when?

The analysis reads a cache, so "today" is not the answer. The figures describe the
data as it stood when it was fetched, and that date is stamped on every cached
response — it is what the memo quotes.

In [2]:
fetch_log = load_fetch_log(PROCESSED / "fetch_log.csv")
dates = fetch_log["fetched_at"].astype(str).str.slice(0, 10)
print("data as of:", dates.min(), "to", dates.max() if dates.max() != dates.min() else "(single day)")
print("films fetched:", int((fetch_log["http_status"] == 200).sum()))

data as of: 2026-11-19 to (single day)
films fetched: 112


## Stage 1 — every title on the list, and what happened to it

The ledger holds **one row per row of the source file, forever**. Nothing is ever
dropped; a row that cannot be used gets a status and a reason. This is what makes the
memo's "127 received, 112 analysed" sentence checkable rather than asserted.

In [3]:
ledger = load_ledger(PROCESSED / "id_ledger.csv")
print(f"{len(ledger)} rows in the ledger ({ledger.attrs.get('data_rows', len(ledger) - 1)} data rows)")
reconciliation_summary(ledger)

128 rows in the ledger (127 data rows)


,status,rows
0,fetched,112
1,duplicate,6
2,not_found_404,5
3,invalid_id,3
4,duplicate_other_namespace,1


In [4]:
# The rows that did not make it, with the reason recorded on each.
unusable = ledger.loc[~ledger["status"].isin(["fetched", "header_row"])]
unusable[["source_row", "raw_value", "status", "reason"]]

,source_row,raw_value,status,reason
17,18,8471293,not_found_404,TMDb has no film under this id (404)
29,30,-1,invalid_id,placeholder '-1' — a human's 'no data'
34,35,9412887,not_found_404,TMDb has no film under this id (404)
42,43,603,duplicate,same id as source_row 40
59,60,278,duplicate,same id as source_row 27
68,69,NaN,invalid_id,blank row
72,73,8471294,not_found_404,TMDb has no film under this id (404)
89,90,1891,duplicate,same id as source_row 55
90,91,155,duplicate,same id as source_row 2
92,93,9412888,not_found_404,TMDb has no film under this id (404)


### The one that is worth dwelling on

Row 95 holds `tt0468569` — an IMDb identifier in a column of TMDb ids. The obvious fix
is to resolve it and add the film back. It resolves to *The Dark Knight*, **which is
already on the list twice**, so recovering it would have put a third copy of one
franchise film into every comparison.

The procedure that handles this is in `fetch.resolve_foreign_ids`, so next quarter's
file gets the same treatment without anyone remembering this paragraph.

## Stages 3 and 4 — four tables, one grain each

A grain mistake never raises. It produces a plausible number: a sum over the cast
table counts a film once per actor. So each table's grain is asserted, not assumed.

In [5]:
tables = {name: pd.read_csv(INTERIM / f"{name}.csv")
          for name in ("films", "cast", "crew", "genres")}
pd.DataFrame([
    {"table": name, "rows": len(frame), "columns": len(frame.columns)}
    for name, frame in tables.items()
])

,table,rows,columns
0,films,112,20
1,cast,7827,7
2,crew,24932,6
3,genres,320,3


In [6]:
# The join is proven rather than trusted: validate="one_to_one" and indicator=True
# on every merge, with the counts written out.
pd.read_csv(PROCESSED / "join_report.csv")

,merge,left_rows,right_rows,result_rows,matched,left_only,right_only
0,directors,112,112,112,112,0,0
1,leads,112,112,112,112,0,0
2,genres,112,112,112,112,0,0


## Stage 5 — real extremes are not errors

*Star Wars* returned 70 times its budget. A budget of 5 would be a typo. Both are far
outside anything typical, and no threshold on its own separates them — so every flagged
value carries a **verdict and a reason**, and nothing is deleted.

In [7]:
flags = pd.read_csv(PROCESSED / "flags.csv")
print("verdicts:", flags["verdict"].value_counts().to_dict())
flags[["title", "flag", "verdict", "why"]].head(10)

verdicts: {'real_extreme': 8}


,title,flag,verdict,why
0,Star Wars,roi_extreme,real_extreme,"ROI 70.5x, a 1977 film: a low nominal budget a..."
1,Raiders of the Lost Ark,roi_extreme,real_extreme,"ROI 21.7x, a 1981 film: a low nominal budget a..."
2,Back to the Future,roi_extreme,real_extreme,"ROI 20.6x, a 1985 film: a low nominal budget a..."
3,The Godfather,roi_extreme,real_extreme,"ROI 40.8x, a 1972 film: a low nominal budget a..."
4,"The Good, the Bad and the Ugly",roi_extreme,real_extreme,"ROI 32.4x, a 1966 film: a low nominal budget a..."
5,Jaws,roi_extreme,real_extreme,"ROI 67.2x, a 1975 film: a low nominal budget a..."
6,Pulp Fiction,roi_extreme,real_extreme,"ROI 26.7x, a 1994 film: a low nominal budget a..."
7,The Empire Strikes Back,roi_extreme,real_extreme,"ROI 29.9x, a 1980 film: a low nominal budget a..."


**Every one of the eight extremes is pre-2000.** That is why they are flagged and kept
rather than clipped: winsorising would have removed 20% of the pre-2000 films and none
of the modern ones — an "outlier treatment" that only ever fires on one era is a thumb
on the scale of the comparison Finance is already sceptical about.

In [8]:
# Why pre- and post-2000 money is not the same unit.
pd.read_csv(PROCESSED / "era_comparability.csv")

,era,films,median_budget,median_revenue,median_roi
0,2000+,71,150000000.0,738599701.0,4.923998
1,pre-2000,41,30000000.0,321365567.0,9.539255


## Stage 6 — does the answer depend on the judgement calls?

Each cleaning choice is a setting, so the rejected alternative is *runnable*. This
table re-runs the headline comparison under each one. Read `franchise_ahead` first
("is the recommendation safe?") and `gap` second ("how precisely may I state it?").

In [9]:
pd.read_csv(PROCESSED / "sensitivity.csv")

,parameter,value,films_in_money_metrics,franchise_n,franchise_median_roi,standalone_n,standalone_median_roi,franchise_ahead,gap
0,performance.treat_zero_money_as_missing,true (default),112,87,5.526,25,5.000,True,0.526
1,performance.treat_zero_money_as_missing,false,112,87,5.526,25,5.000,True,0.526
2,franchise.min_members_on_list,2 — lone members are standalone (default),112,87,5.526,25,5.000,True,0.526
3,franchise.min_members_on_list,1 — lone members are franchise,112,96,5.624,16,3.845,True,1.779
4,performance.min_budget_for_roi,"1,000,000 (default)",112,87,5.526,25,5.000,True,0.526
5,performance.min_budget_for_roi,"10,000,000",105,85,5.451,20,4.762,True,0.689


The direction holds in every run. The **size** of the lead swings 3.4x, driven almost
entirely by where the nine lone collection members go — which is why the memo gives a
range and names the choice it depends on.

## Stage 7 — the comparison, both windows

Counts beside every statistic. That matters more than usual here: the standalone group
is 25 films, and 12 after 2000.

In [10]:
for window in ("all_years", "modern"):
    print(f"--- {window} ---")
    display(pd.read_csv(PROCESSED / f"kpi_{window}.csv")[[
        "franchise_group_strict", "films", "median_roi", "floor_roi",
        "iqr_roi", "mean_roi", "hit_rate_primary",
    ]])

--- all_years ---


,franchise_group_strict,films,median_roi,floor_roi,iqr_roi,mean_roi,hit_rate_primary
0,franchise,87,5.526,3.906,4.388,7.936,0.954
1,standalone,25,5.000,2.641,11.714,10.480,0.800


--- modern ---


,franchise_group_strict,films,median_roi,floor_roi,iqr_roi,mean_roi,hit_rate_primary
0,franchise,59,5.052,3.821,2.470,5.447,0.949
1,standalone,12,4.488,2.858,2.297,4.894,0.917


### The line that looks like a formatting preference

Compare `mean_roi` with `median_roi` above. **On the mean, standalones win. On the
median, franchises win.** The three exceptional pre-2000 films sit disproportionately
in the smaller group, so the mean reverses the recommendation. The memo reports medians
and says so unprompted.

In [11]:
# Does the answer change between windows? Every measure, not just the median.
pd.read_csv(PROCESSED / "kpi_flip_check.csv")

,measure,better_is,winner_all_years,winner_modern,flipped
0,median_roi,higher,franchise,franchise,False
1,floor_roi,higher,franchise,franchise,False
2,hit_rate_primary,higher,franchise,franchise,False
3,iqr_roi,lower,franchise,standalone,True


`iqr_roi` is marked `better_is: lower` — a tighter group is the safer buy. It is the
one measure that flips, and the margin is 0.17 on 12 films, so the defensible statement
is that the franchise spread advantage *disappears* after 2000, not that standalones
became safer.

In [12]:
# Inside a franchise: are the middle entries the weak ones?
pd.read_csv(PROCESSED / "kpi_position_in_franchise.csv")

,position,films,collections,median_roi,floor_roi,median_revenue,median_rating
0,first,22,22,7.759,5.096,454206770.5,7.872
1,middle,41,22,5.412,3.834,660998756.0,7.339
2,last,22,22,4.438,3.531,769618450.0,7.080


Pooled, that looks like a 43% decline from opener to final entry. But openers are
disproportionately older films, and older films look more profitable for reasons that
have nothing to do with quality. Within one era the decline is **14%**, and middle
entries sit *above* last entries — so the team is right about the direction and wrong
about which entries are weak.

In [13]:
# The shortlist, with the runners-up it beat.
display(pd.read_csv(PROCESSED / "shortlist_franchises.csv")[[
    "collection_name", "films_on_list", "median_roi", "floor_roi", "bundle_roi", "selected",
]])
display(pd.read_csv(PROCESSED / "shortlist_standalones.csv")[[
    "title", "release_year", "roi", "revenue", "franchise_status", "selected",
]])

,collection_name,films_on_list,median_roi,floor_roi,bundle_roi,selected
0,Star Wars Collection,6,13.402,5.412,10.828,True
1,The Lord of the Rings Collection,3,11.725,9.370,10.964,True
2,Indiana Jones Collection,4,10.888,4.252,7.111,True
3,Jurassic Park Collection,3,8.475,3.965,8.934,False
4,Back to the Future Collection,3,8.322,6.127,9.793,False
5,Ice Age Collection,3,8.262,6.496,8.432,False


,title,release_year,roi,revenue,franchise_status,selected
0,Jaws,1975,67.236,470653000,lone_collection_member,True
1,"The Good, the Bad and the Ugly",1966,32.417,38900000,standalone,True
2,Pulp Fiction,1994,26.741,213928762,standalone,True
3,The Lion King,1994,16.966,763455561,lone_collection_member,True
4,Schindler's List,1993,14.608,321365567,standalone,True
5,Spirited Away,2001,14.470,274925095,standalone,False
6,The Silence of the Lambs,1991,14.355,272742922,lone_collection_member,False
7,Forrest Gump,1994,12.316,677387716,standalone,False


Two of the five "standalone" picks are **lone collection members** — films that belong
to a franchise of which only one instalment is on offer. They are legitimate picks, but
presenting *Jaws* as a standalone without that qualification invites a fair objection,
so the label travels with the row.

In [14]:
# Talent: the film count stays visible on every row, below the minimum as well as above.
directors = pd.read_csv(PROCESSED / "kpi_talent_directors.csv")
directors.loc[directors["meets_min_films"], [
    "person_name", "films", "median_roi", "floor_roi", "hit_rate_roi",
    "above_overall_median",
]]

,person_name,films,median_roi,floor_roi,hit_rate_roi,above_overall_median
0,Steven Spielberg,8,13.252,4.252,1.000,True
1,Robert Zemeckis,4,10.319,6.127,1.000,True
2,James Cameron,3,9.909,5.077,1.000,True
3,George Lucas,4,8.311,5.412,1.000,True
4,Francis Ford Coppola,3,7.892,2.533,1.000,True
5,Peter Jackson,6,6.727,3.824,1.000,True
6,Quentin Tarantino,3,6.030,5.072,1.000,True
7,David Yates,4,5.036,3.736,1.000,False
8,Lilly Wachowski,3,4.924,2.833,1.000,False
9,Lana Wachowski,3,4.924,2.833,1.000,False


In [15]:
# And the sensitivity: does the reliable list survive widening from leads to top-3 billed?
widened = pd.read_csv(PROCESSED / "kpi_talent_sensitivity.csv")
print(f"{len(widened)} reliable names; "
      f"{int(widened['appears_only_when_widened'].sum())} appear only when widened")
widened.head(10)

44 reliable names; 25 appear only when widened


,person_id,person_name,films_as_lead,median_roi_as_lead,reliable_as_lead,films_top_3,median_roi_top_3,reliable_top_3,appears_only_when_widened
0,2,Mark Hamill,3.0,29.911,True,3,29.911,True,False
1,4,Carrie Fisher,NaN,NaN,False,3,29.911,True,True
2,3,Harrison Ford,5.0,9.879,True,8,14.800,True,False
3,109,Elijah Wood,3.0,11.725,True,3,11.725,True,False
4,110,Viggo Mortensen,NaN,NaN,False,3,11.725,True,True
5,3896,Liam Neeson,2.0,11.854,False,3,9.100,True,True
6,521,Michael J. Fox,3.0,8.322,True,3,8.322,True,False
7,1062,Christopher Lloyd,NaN,NaN,False,3,8.322,True,True
8,15757,Ray Romano,3.0,8.262,True,3,8.262,True,False
9,5723,John Leguizamo,NaN,NaN,False,3,8.262,True,True


## Stage 8 — where the time goes, and what it will cost at 500 titles

The boundary, stated before any number: **stages 3 to 7, warm cache, one machine,
single-threaded. Acquisition excluded** — timing it would measure the network, not this
code — and reported beside the budget rather than inside it.

In [16]:
display(pd.read_csv(PROCESSED / "time_budget.csv"))
display(pd.read_csv(PROCESSED / "projection.csv"))

,stage,seconds,share_of_analysis,in_budget,note
0,extract.read_ids,0.053,NaN,False,"outside the stated boundary, reported for comp..."
1,extract.acquire,4.393,NaN,False,"excluded: network-bound and cached, so timing ..."
2,extract.shape,1.700,0.655,True,NaN
3,validate,0.280,0.108,True,NaN
4,transform,0.615,0.237,True,NaN


,stage,measured_films,measured_seconds,exponent_k,scaling,fit_quality,projected_films,projected_seconds
0,extract.shape,112,0.726,0.98,roughly linear,measured,500,3.15
1,validate,112,0.112,0.05,sub-linear (fixed costs dominate at this size),measured,500,0.12
2,transform,112,0.149,1.00,roughly linear,assumed linear (measured exponent was negative),500,0.67


The projection is **measured, not multiplied**: stages 3–7 are re-run over subsets of
the real payloads at four sizes and a scaling exponent is fitted per stage. The
all-linear figure is published beside the fitted one so the conclusion does not rest on
the fit being right.

## Stage 9 — checking our own work against the live source

A random sample of 12 from 112 would usually miss every film that matters. So the
sample is stratified: the shortlisted titles, **all** the extremes, Yaw's window, and an
unbiased control.

In [17]:
display(pd.read_csv(PROCESSED / "spotcheck_sample.csv"))
display(pd.read_csv(PROCESSED / "spotcheck_summary.csv"))

,tmdb_id,title,slice,why_this_slice
0,11,Star Wars,shortlisted,named in the recommendation; an error here is ...
1,1891,The Empire Strikes Back,shortlisted,named in the recommendation; an error here is ...
2,1895,Star Wars: Episode III - Revenge of the Sith,shortlisted,named in the recommendation; an error here is ...
3,85,Raiders of the Lost Ark,flagged_extreme,ROI above 20x: the most influential rows in th...
4,105,Back to the Future,flagged_extreme,ROI above 20x: the most influential rows in th...
5,238,The Godfather,flagged_extreme,ROI above 20x: the most influential rows in th...
6,429,"The Good, the Bad and the Ugly",flagged_extreme,ROI above 20x: the most influential rows in th...
7,578,Jaws,flagged_extreme,ROI above 20x: the most influential rows in th...
8,680,Pulp Fiction,flagged_extreme,ROI above 20x: the most influential rows in th...
9,13,Forrest Gump,pre_2000,"the window Finance distrusts, and where money ..."


,slice,verdict,fields,films
0,flagged_extreme,expected_drift,9,6
1,flagged_extreme,identical,45,6
2,pre_2000,expected_drift,3,3
3,pre_2000,identical,24,3
4,random_control,expected_drift,3,3
5,random_control,identical,24,3
6,shortlisted,expected_drift,3,3
7,shortlisted,identical,24,3


What this establishes is **fidelity** — that the values the analysis used are the values
the source holds. It does *not* establish that the source is correct, and it is a weak
test of drift over time, because the cache was fetched hours before the check. The full
write-up, generated by the stage itself, is in
[`reports/spotcheck.md`](../reports/spotcheck.md).

## Stage 10 — the gate, then the memo

Four checks run before the memo is written. If any fails, **no memo and no shortlist**
are produced. Adjoa reruns this quarterly and sometimes live; the failure this protects
against is a confident number from a broken run, in front of a board, with nothing to
signal that it is wrong.

In [18]:
pd.read_csv(PROCESSED / "gate.csv")

,check,passed,detail
0,every source row accounted for,True,"0 row(s) unaccounted for, limit 0. The memo qu..."
1,enough films to compare groups,True,"112 films analysed, minimum 40. Below this the..."
2,money data complete enough to decide on,True,0% of films have no usable budget or box offic...
3,live re-check does not change the answer,True,the live re-check leaves the recommendation un...


## Where to read next

| | |
|---|---|
| [`reports/memo.md`](../reports/memo.md) | **The answer.** One page, business language. |
| [`reports/data_diagnosis.md`](../reports/data_diagnosis.md) | Every problem found, ranked by how much it moves the recommendation. |
| [`decisions/`](../decisions/) | Every judgement call in five parts, including what it costs if it is wrong. |
| [`reports/spotcheck.md`](../reports/spotcheck.md) | The live re-check. |
| [`reports/improvement.md`](../reports/improvement.md) | The one addition beyond the brief, and why that one. |